In [27]:
import os
from dotenv import load_dotenv
from ultralytics.data.converter import convert_coco
import yaml
import glob
import shutil
from ultralytics import YOLO

# force Python to read the .env file 
load_dotenv()

True

In [35]:
# define dataset parent directory
data_dir = os.path.abspath("../data/all_cropped")

This cell is used to convert, reorganize, and create necessary files to run YOLO models. It expects train and val folders in the ML dataset directory which each contain their own COCO formatted JSON files.

In [38]:
# define the directory containing ML ready tiles
ml_data_dir = os.path.join(data_dir, 'ml_dataset')

# convert the COCO formatted JSONs to YOLO format
print("Converting train data...")
convert_coco(
    labels_dir=os.path.join(ml_data_dir, 'train'), 
    use_segments=True,
    save_dir=os.path.join(ml_data_dir, 'train', 'labels') # FIX: Force output location
)

print("Converting val data...")
convert_coco(
    labels_dir=os.path.join(ml_data_dir, 'val'), 
    use_segments=True,
    save_dir=os.path.join(ml_data_dir, 'val', 'labels') # FIX: Force output location
)
print("Converted files to YOLO format\n")

# reorganize the label files into the correct structure for YOLO
print("Reorganizing label files into correct YOLO folders...")

for split in ['train', 'val']:
    labels_target_dir = os.path.join(ml_data_dir, split, "labels")
    os.makedirs(labels_target_dir, exist_ok=True)
    
    # search for all .txt files anywhere inside the split folder
    all_txts = glob.glob(os.path.join(ml_data_dir, split, "**", "*.txt"), recursive=True)
    
    moved_count = 0
    for txt_path in all_txts:
        # ignore YOLO's classes summary file
        if os.path.basename(txt_path) == "classes.txt":
            continue
            
        # move it if it isn't already in the target 'labels' folder
        if os.path.dirname(txt_path) != os.path.abspath(labels_target_dir):
            shutil.move(txt_path, os.path.join(labels_target_dir, os.path.basename(txt_path)))
            moved_count += 1
            
    print(f"Moved {moved_count} label files to {split}/labels/")

# create YOLO .yaml configuration file
yolo_yaml_path = os.path.join(ml_data_dir, "chaos_blocks.yaml")

yolo_config = {
    # provide the path to the dataset
    "path": os.path.abspath(ml_data_dir),
    
    # YOLO looks for a 'labels' folder next to these 'images' folders
    "train": "train/images",
    "val": "val/images",
    
    # class dictionary
    "names": {
        0: "chaos_block"
    }
}

with open(yolo_yaml_path, 'w') as f:
    yaml.dump(yolo_config, f, default_flow_style=False)

print(f"Created YOLO config file at: {yolo_yaml_path}")

Converting train data...
Annotations /Users/jjwill18/Desktop/europa-chaos-blocks/data/all_cropped/ml_dataset/train/train_coco.json: 100% ━━━━━━━━━━━━ 1368/1368 1.6Kit/s 0.9s0.0s
COCO data converted successfully.
Results saved to /Users/jjwill18/Desktop/europa-chaos-blocks/data/all_cropped/ml_dataset/train/labels-2
Converting val data...
Annotations /Users/jjwill18/Desktop/europa-chaos-blocks/data/all_cropped/ml_dataset/val/val_coco.json: 100% ━━━━━━━━━━━━ 138/138 1.9Kit/s 0.1s
COCO data converted successfully.
Results saved to /Users/jjwill18/Desktop/europa-chaos-blocks/data/all_cropped/ml_dataset/val/labels-2
Converted files to YOLO format

Reorganizing label files into correct YOLO folders...
Moved 1368 label files to train/labels/
Moved 138 label files to val/labels/
Created YOLO config file at: /Users/jjwill18/Desktop/europa-chaos-blocks/data/all_cropped/ml_dataset/chaos_blocks.yaml


In [39]:
# define absolute paths to the yaml file and run results directory
# define the directory containing ML ready tiles
ml_data_dir = os.path.join(data_dir, 'ml_dataset')
yaml_path = ml_data_dir = os.path.join(ml_data_dir, 'chaos_blocks.yaml')
results_dir = os.path.abspath("../results/runs")

# define where you want the base model to be saved/loaded from
model_dir = os.path.abspath("../models/")
os.makedirs(model_dir, exist_ok=True)

# using the pretrained model here
model_path = os.path.join(model_dir, "yolo26s-seg.pt")

print(f"Base model location: {model_path}")
print(f"Saving runs to: {results_dir}")

# load the model
model = YOLO(model_path) 

results = model.train(
    data=yaml_path,
    epochs=50,
    patience=15,  # if mAP doesn't improve for 15 epochs, stop early
    imgsz=512,    # up sample the image to 512
    batch=8,      # decrease batch size to compensate for upsampled image
    device='mps',           # hardware for running
    project=results_dir,    # define directory to store the results
    name="run_12_yolo26s",  # folder to save results
    lr0=0.001,           # Small, careful learning steps
    cos_lr=True,         # Smooth learning curve
    weight_decay=0.001,  # Strict penalty for overfitting
    overlap_mask=False,     # False means model will not automatically merge overlapping boxes
    iou=0.7,                # IoU threshold (loswer = more aggressive suppression)
    box=7.5,                # Multiplier for bounding box loss 
    hsv_h=0.0,           # Disable Hue shifts (Keep it grayscale)
    hsv_s=0.0,           # Disable Saturation shifts
    hsv_v=0.6,           # Aggressive Brightness shifts (Shadow resilience)
    bgr=0.2              # 20% chance of Blur
)

Base model location: /Users/jjwill18/Desktop/europa-chaos-blocks/models/yolo26s-seg.pt
Saving runs to: /Users/jjwill18/Desktop/europa-chaos-blocks/results/runs
Ultralytics 8.4.174 🚀 Python-3.11.16 torch-2.13.0 MPS (Apple M3 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.2, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/Users/jjwill18/Desktop/europa-chaos-blocks/data/all_cropped/ml_dataset/chaos_blocks.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.6, imgsz=512, iou=0.7, kobj=1.0, line_width=

/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      7.98G      2.345      3.671      2.816   0.008797      3.929        140        512: 100% ━━━━━━━━━━━━ 173/173 1.8it/s 1:340.6sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.5it/s 5.8s0.7ss
                   all        138       3462     0.0983      0.167      0.048     0.0168      0.101      0.166      0.048     0.0137

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
       2/50      6.89G      2.448      3.672      2.362   0.008816      3.283        227        512: 0% ──────────── 0/173  1.0s

/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50       6.9G      2.273      3.438      2.405   0.008107      3.548        118        512: 100% ━━━━━━━━━━━━ 173/173 1.9it/s 1:300.6sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.5it/s 6.1s0.7ss
                   all        138       3462      0.204      0.239      0.138     0.0538      0.194      0.215      0.121     0.0376

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
       3/50      6.87G      2.237      3.326      2.511   0.008606      3.247        185        512: 0% ──────────── 0/173  1.0s

...

      29/50      6.84G      1.959      2.775      1.619   0.006881      1.989         69        512: 100% ━━━━━━━━━━━━ 173/173 1.9it/s 1:330.6sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.7it/s 5.4s0.6ss
                   all        138       3462      0.337      0.274      0.211     0.0871      0.317      0.263      0.187     0.0602

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      30/50      6.86G      2.025      2.803      1.743   0.006928      2.139        183        512: 0% ──────────── 0/173  1.1s

/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/50      6.84G      1.966      2.763      1.567   0.006735      1.902        195        512: 100% ━━━━━━━━━━━━ 173/173 1.9it/s 1:320.6sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.6it/s 5.5s0.6ss
                   all        138       3462      0.312      0.347       0.23     0.0899      0.299       0.33       0.21     0.0654

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
      31/50      6.86G      1.986      2.527       1.55   0.006559      1.957        324        512: 0% ──────────── 0/173  1.2s

/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      31/50      6.81G      1.969      2.757       1.58   0.006905      1.951         99        512: 100% ━━━━━━━━━━━━ 173/173 1.9it/s 1:320.6sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.6it/s 5.6s0.6ss
                   all        138       3462      0.328      0.309      0.216     0.0853      0.315      0.285      0.191     0.0567
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 16, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

31 epochs completed in 0.928 hours.
Optimizer stripped from /Users/jjwill18/Desktop/europa-chaos-blocks/results/runs/run_12_yolo26s-5/weights/last.pt, 23.3MB
Optimizer stripped from /Users/jjwill18/Desktop/europa-chaos-blocks/results/runs/run_12_yolo26s-5/weights/best